In [50]:
import numpy as np
import pandas as pd

# Load the datasets created by data_analysis.ipynb.
processed_data = np.load("data/processed_datasets.npz")
X_train = pd.DataFrame(processed_data["X_train"])
X_test = pd.DataFrame(processed_data["X_test"])
Y_train = pd.Series(processed_data["Y_train"], name="churn")
Y_test = pd.Series(processed_data["Y_test"], name="churn")
processed_data.close()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Y_train:", Y_train.shape)
print("Y_test:", Y_test.shape)

X_train: (659028, 44)
X_test: (164758, 44)
Y_train: (659028,)
Y_test: (164758,)


In [42]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

In [43]:
models = {
    "dummy": DummyClassifier(
        strategy="most_frequent"
    ),

    "logistic_regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
}

In [44]:
feature_configs = {
    "all_features": None, 
    "top_30": 30,
    "top_25": 25,
    "top_20": 20,
    "top_15": 15,
    "top_10": 10,
}

In [45]:
def create_pipeline(model, k=None):
    if isinstance(model, DummyClassifier):
        feature_selector = "passthrough"
    elif k is None:
        feature_selector = "passthrough"
    else:
        feature_selector = SelectKBest(
            score_func=f_classif,
            k=k
        )

    pipeline = Pipeline([
        ("feature_selection", feature_selector),
        ("model", model)
    ])

    return pipeline

In [49]:
pipelines = {}

for model_name, model in models.items():

    for feature_name, k in feature_configs.items():

        if model_name == "dummy" and feature_name != "all_features":
            continue

        pipeline_name = f"{model_name}_{feature_name}"

        pipelines[pipeline_name] = create_pipeline(
            model,
            k
        )

print("\n".join(pipelines.keys()))

dummy_all_features
logistic_regression_all_features
logistic_regression_top_30
logistic_regression_top_25
logistic_regression_top_20
logistic_regression_top_15
logistic_regression_top_10
random_forest_all_features
random_forest_top_30
random_forest_top_25
random_forest_top_20
random_forest_top_15
random_forest_top_10
xgboost_all_features
xgboost_top_30
xgboost_top_25
xgboost_top_20
xgboost_top_15
xgboost_top_10


In [55]:
import mlflow
import mlflow.sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Use a fresh local SQLite store with the current MLflow schema.
mlflow.set_tracking_uri("sqlite:///mlflow_churn.db")
mlflow.set_experiment("churn-models")


def evaluate_model(model, X, y):
    predictions = model.predict(X)
    metrics = {
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
    }

    if hasattr(model, "predict_proba"):
        metrics["roc_auc"] = roc_auc_score(y, model.predict_proba(X)[:, 1])

    return metrics


results = []

In [59]:
# Validate MLflow serialization with a small pipeline before the full experiment.
serialization_test = pipelines["logistic_regression_top_10"]
serialization_test.fit(X_train.iloc[:1000], Y_train.iloc[:1000])
with mlflow.start_run(run_name="serialization_check"):
    mlflow.sklearn.log_model(
        serialization_test,
        "model",
        serialization_format="cloudpickle"
    )
print("MLflow model serialization check passed.")

/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [23] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/home/rami/miniconda3/envs/mlops/lib/python3.14/site-packages/sklearn/feature_selection/_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
2026/09/24 15:51:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/09/24 15:51:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


MLflow model serialization check passed.


In [60]:
for pipeline_name, pipeline in pipelines.items():
    feature_name = next(
        name for name in feature_configs if pipeline_name.endswith(f"_{name}")
    )
    model_name = pipeline_name[:-(len(feature_name) + 1)]
    model = pipeline.named_steps["model"]
    k = feature_configs[feature_name]

    with mlflow.start_run(run_name=pipeline_name):
        pipeline.fit(X_train, Y_train)
        metrics = evaluate_model(pipeline, X_test, Y_test)

        mlflow.log_param("model_type", model_name)
        mlflow.log_param("feature_set", feature_name)
        mlflow.log_param("n_features", X_train.shape[1] if k is None else k)

        model_params = {
            f"model_{key}": value
            for key, value in model.get_params().items()
        }
        mlflow.log_params(model_params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            pipeline,
            "model",
            serialization_format="cloudpickle"
        )

        results.append({
            "model": model_name,
            "feature_set": feature_name,
            **metrics
        })

2026/09/24 15:51:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/24 15:51:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/09/24 15:51:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/24 15:51:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p